<a href="https://colab.research.google.com/github/shuchivardan/Shuchi_V_Portfolio/blob/main/notebooks/Final_Decoder_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import os

token = userdata.get('GH_TOKEN')
repo  = "csugasi/DATASCI266-Banking77-Encoder-vs-Decoder"
path  = "/content/repo"

if not os.path.exists(path):
    !git clone -q -b shuchi/dev https://{token}@github.com/{repo}.git {path}

%cd {path}
!nvidia-smi

/content/repo
Wed Nov 18 04:16:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------------

In [2]:
!pip install trl transformers datasets accelerate peft -q

In [3]:
import pandas as pd, json
from datasets import Dataset
import sys
sys.path.insert(0, "scripts")
from predictions import match_intent, write_predictions
import time
import glob
import numpy as np
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [4]:
train = pd.read_parquet("data/splits/full_train.parquet")
val   = pd.read_parquet("data/splits/val.parquet")
labels = json.load(open("data/raw/label_names.json"))

print("train:", train.shape, "| val:", val.shape, "| labels:", len(labels))

train: (9618, 3) | val: (385, 3) | labels: 77


In [5]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
# Setup Cell

#!sed -i 's/    "qwen_qlora": "decoder",/    "qwen_qlora": "decoder",\n    "qwen_sft": "decoder",/' scripts/evaluation.py
!grep -n '"qwen' scripts/evaluation.py

import sys, os, time, torch
import numpy as np, pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer
sys.path.insert(0, "scripts")
from predictions import match_intent, write_predictions
import evaluation as ev
ev.MODEL_TYPE["qwen_sft"] = "decoder"      # register in the loaded module (file on disk already has it)
assert "qwen_sft" in ev.MODEL_TYPE, ev.__file__

SYSTEM = ("You classify banking customer messages. Reply with exactly one intent "
          "name from the list, copied exactly, and nothing else.")
INTENT_LIST = "\n".join(labels)

def user_text(message):
    return f"Intents:\n{INTENT_LIST}\n\nMessage: {message}\n\nIntent:"

def format_example(row):
    return {
        "prompt": [{"role": "system", "content": SYSTEM},
                   {"role": "user", "content": user_text(row["text"])}],
        "completion": [{"role": "assistant", "content": labels[row["label"]]}],
    }

def build_prompt(message):
    chat = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_text(message)}]
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

def classify(messages, batch_size=16):
    outputs = []
    for i in range(0, len(messages), batch_size):
        prompts = [build_prompt(m) for m in messages[i:i + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=20, do_sample=False)
        outputs += tokenizer.batch_decode(generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return outputs

test_df = pd.read_parquet("data/raw/test.parquet")
val_texts, val_labels = val["text"].tolist(), val["label"].to_numpy()
test_texts, test_labels = test_df["text"].tolist(), test_df["label"].to_numpy()

lens = [len(tokenizer(build_prompt(t)).input_ids) for t in train["text"]]
print("max prompt tokens:", max(lens))

47:    "qwen_zeroshot": "decoder",
48:    "qwen_qlora": "decoder",
49:    "qwen_sft": "decoder",
50:    "qwen_sft": "decoder",
61:ZERO_SHOT_MODELS = ("qwen_zeroshot", "frontier")  # the only models allowed the 'zeroshot' budget
max prompt tokens: 524


In [8]:
!grep -c '"qwen_sft"' scripts/evaluation.py

2


In [9]:
!git checkout scripts/evaluation.py
!sed -i 's/    "qwen_qlora": "decoder",/    "qwen_qlora": "decoder",\n    "qwen_sft": "decoder",/' scripts/evaluation.py
!grep -c '"qwen_sft"' scripts/evaluation.py

Updated 1 path from the index
1


In [12]:
# to restore where the code froze:
!git checkout scripts/evaluation.py
!sed -i 's/    "qwen_qlora": "decoder",/    "qwen_qlora": "decoder",\n    "qwen_sft": "decoder",/' scripts/evaluation.py
!grep -c '"qwen_sft"' scripts/evaluation.py

Updated 1 path from the index
1


In [11]:
# check learning rate on new set-up

PILOT_LRS = [2e-5, 5e-5, 1e-4]
PILOT_SEEDS = [0,1,2]
pilot = []

for lr in PILOT_LRS:
    for seed in PILOT_SEEDS:
        tr = pd.read_parquet(f"data/splits/train_k10_seed{seed}.parquet")
        tr_ds = Dataset.from_list([format_example(r) for _, r in tr.iterrows()])
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda")
        cfg = SFTConfig(
            output_dir=f"./pilot/{lr}_seed{seed}",
            num_train_epochs=3, per_device_train_batch_size=4, gradient_accumulation_steps=4,
            learning_rate=lr, bf16=True, logging_steps=20, save_strategy="no",
            report_to="none", max_length=1024, seed=seed,
        )
        t = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=tr_ds, args=cfg)
        t.train()
        model.eval()
        pred = np.array([match_intent(r) for r in classify(val_texts)])
        acc = (pred == val_labels).mean()
        print(f"lr={lr} seed{seed}: val acc={acc:.3f} | unmatched={(pred == -1).mean():.3f}")
        pilot.append({"lr": lr, "seed": seed, "val_acc": acc})
        del model, t; torch.cuda.empty_cache()

print(pd.DataFrame(pilot).groupby("lr").val_acc.mean())

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Step,Training Loss
20,0.403387
40,0.229447
60,0.140532
80,0.087900
100,0.079076
120,0.067559
140,0.042998


... (47 outputs left out)

Truncating train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Step,Training Loss
20,1.265822
40,0.369918


KeyboardInterrupt: 

In [13]:
del model, t
torch.cuda.empty_cache()

NameError: name 'model' is not defined

In [17]:
from google.colab import drive
drive.mount("/content/drive")
BACKUP = "/content/drive/MyDrive/qwen_sft_backup"
os.makedirs(BACKUP, exist_ok=True)
!cp results/qwen_sft_*.csv {BACKUP}/ 2>/dev/null   # back up the k1 seed0 files already done

MODEL_NAME = "qwen_sft"
LEARNING_RATE, EPOCHS, MAX_LEN = 5e-5, 3, 1024
BUDGETS = ["k1", "k3", "k5", "k10", "k20", "k30", "p50", "full"]
SEEDS = [0, 1, 2]

def budget_file(budget, seed):
    if budget == "full":
        return "data/splits/full_train.parquet"
    return f"data/splits/train_{budget}_seed{seed}.parquet"

for budget in BUDGETS:
    for seed in SEEDS:
        if os.path.exists(f"results/{MODEL_NAME}_{budget}_seed{seed}_test.csv"):
            print(f"skip {budget} seed{seed} (done)"); continue
        print(f"\n===== {budget} seed{seed} =====")

        tr = pd.read_parquet(budget_file(budget, seed))
        tr_ds = Dataset.from_list([format_example(r) for _, r in tr.iterrows()])
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda")

        cfg = SFTConfig(
            output_dir=f"./runs/{budget}_seed{seed}",
            num_train_epochs=EPOCHS, per_device_train_batch_size=4, gradient_accumulation_steps=4,
            learning_rate=LEARNING_RATE, bf16=True, logging_steps=20, save_strategy="no",
            report_to="none", max_length=MAX_LEN, seed=seed,
        )
        t = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=tr_ds, args=cfg)

        b = t.data_collator([t.train_dataset[0]])
        print("trains on:", repr(tokenizer.decode(b["input_ids"][0][b["labels"][0] != -100])))

        start = time.time(); t.train(); mins = (time.time() - start) / 60
        model.eval()
        for split, texts, y in [("val", val_texts, val_labels), ("test", test_texts, test_labels)]:
            raw = classify(texts)
            pred = np.array([match_intent(r) for r in raw])
            write_predictions(MODEL_NAME, budget, seed, split, pred, raw_outputs=raw)
            print(f"{split} acc={(pred == y).mean():.3f} | unmatched={(pred == -1).mean():.3f}")
        print(f"train time {mins:.1f} min, examples {len(tr_ds)}")

        del model, t; torch.cuda.empty_cache()
        !cp results/{MODEL_NAME}_{budget}_seed{seed}_*.csv {BACKUP}/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
skip k1 seed0 (done)
skip k1 seed1 (done)
skip k1 seed2 (done)
skip k3 seed0 (done)
skip k3 seed1 (done)
skip k3 seed2 (done)
skip k5 seed0 (done)
skip k5 seed1 (done)
skip k5 seed2 (done)
skip k10 seed0 (done)
skip k10 seed1 (done)
skip k10 seed2 (done)
skip k20 seed0 (done)
skip k20 seed1 (done)
skip k20 seed2 (done)
skip k30 seed0 (done)
skip k30 seed1 (done)
skip k30 seed2 (done)
skip p50 seed0 (done)
skip p50 seed1 (done)
skip p50 seed2 (done)
skip full seed0 (done)
skip full seed1 (done)

===== full seed2 =====


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/9618 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/9618 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/9618 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/9618 [00:00<?, ? examples/s]

trains on: 'card_arrival<|im_end|>\n'


Step,Training Loss
20,0.648108
40,0.296673
60,0.220976
80,0.180532
100,0.177914
120,0.144671
140,0.157751
160,0.116748
180,0.126196
200,0.092257


val acc=0.927 | unmatched=0.003
test acc=0.936 | unmatched=0.003
train time 28.5 min, examples 9618


In [18]:
!ls results/qwen_sft_*.csv | wc -l      # must show 48
!git config user.email "shuchivardan@gmail.com"
!git config user.name "Shuchi Vardan"
!git add results/qwen_sft_*.csv scripts/evaluation.py
!git commit -m "Add qwen_sft val+test predictions (intent list in training prompt), all budgets and seeds"
!git push origin shuchi/dev

48
[shuchi/dev 5f5a816] Add qwen_sft val+test predictions (intent list in training prompt), all budgets and seeds
 47 files changed, 79742 insertions(+)
 create mode 100644 results/qwen_sft_full_seed0_test.csv
 create mode 100644 results/qwen_sft_full_seed0_val.csv
 create mode 100644 results/qwen_sft_full_seed1_test.csv
 create mode 100644 results/qwen_sft_full_seed1_val.csv
 create mode 100644 results/qwen_sft_full_seed2_test.csv
 create mode 100644 results/qwen_sft_full_seed2_val.csv
 create mode 100644 results/qwen_sft_k10_seed0_test.csv
 create mode 100644 results/qwen_sft_k10_seed0_val.csv
 create mode 100644 results/qwen_sft_k10_seed1_test.csv
 create mode 100644 results/qwen_sft_k10_seed1_val.csv
 create mode 100644 results/qwen_sft_k10_seed2_test.csv
 create mode 100644 results/qwen_sft_k10_seed2_val.csv
 create mode 100644 results/qwen_sft_k1_seed1_test.csv
 create mode 100644 results/qwen_sft_k1_seed1_val.csv
 create mode 100644 results/qwen_sft_k1_seed2_test.csv
 create mod